# 01 - Data Cleaning & Validation

Cleans and checks `events`, `ads`, `campaigns`, and `users` before any EDA or KPI work.

Core tables for the actual analysis: `events_clean`, `ads_clean`, `campaigns_clean`.

`users_clean` is exported too, but it's not part of the core model - 50 `user_id` values repeat with genuinely conflicting attributes (different age, gender, country, etc.), so it can't be trusted as a relational key.

Rules for this notebook:
- look at the raw data before changing anything
- figure out what's actually wrong before deciding how to fix it
- never invent or rewrite an ID
- check primary keys and foreign keys
- don't filter events by campaign start/end dates - a lot of events fall outside that window and we're not resolving that here
- save each clean table separately instead of one big merged file


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load raw data


In [2]:
# Keep the original source files untouched.
users = pd.read_csv("users.csv")
ads = pd.read_csv("ads.csv")
campaigns = pd.read_csv("campaigns.csv")
events = pd.read_csv("ad_events.csv")

raw_dfs = {
    "USERS": users,
    "ADS": ads,
    "CAMPAIGNS": campaigns,
    "EVENTS": events
}

## 2. Explore before touching anything


In [3]:
def profile_table(df, name):
    print(f"\n{'=' * 75}")
    print(f"{name}")
    print(f"{'=' * 75}")
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nFull-row duplicates:", df.duplicated().sum())
    print("\nUnique values per column:")
    print(df.nunique())
    print("\nSample rows:")
    display(df.head(5))

for name, df in raw_dfs.items():
    profile_table(df, name)


USERS
Shape: 10,000 rows × 7 columns

Columns:
['user_id', 'user_gender', 'user_age', 'age_group', 'country', 'location', 'interests']

Data types:
user_id        object
user_gender    object
user_age        int64
age_group      object
country        object
location       object
interests      object
dtype: object

Missing values:
user_id        0
user_gender    0
user_age       0
age_group      0
country        0
location       0
interests      0
dtype: int64

Full-row duplicates: 0

Unique values per column:
user_id        9950
user_gender       3
user_age         50
age_group         6
country          10
location       7706
interests      1641
dtype: int64

Sample rows:


,user_id,user_gender,user_age,age_group,country,location,interests
0,a2474,Female,24,18-24,United Kingdom,New Mariomouth,"fitness, health"
1,141e5,Male,21,18-24,Germany,Danielsfort,"food, fitness, lifestyle"
2,34db0,Male,27,25-34,Australia,Vincentchester,"fashion, news"
3,20d08,Female,28,25-34,India,Lisaport,"health, news, finance"
4,9e830,Male,28,25-34,United States,Brownmouth,"health, photography, lifestyle"



ADS
Shape: 200 rows × 7 columns

Columns:
['ad_id', 'campaign_id', 'ad_platform', 'ad_type', 'target_gender', 'target_age_group', 'target_interests']

Data types:
ad_id                int64
campaign_id          int64
ad_platform         object
ad_type             object
target_gender       object
target_age_group    object
target_interests    object
dtype: object

Missing values:
ad_id               0
campaign_id         0
ad_platform         0
ad_type             0
target_gender       0
target_age_group    0
target_interests    0
dtype: int64

Full-row duplicates: 0

Unique values per column:
ad_id               200
campaign_id          48
ad_platform           2
ad_type               4
target_gender         3
target_age_group      4
target_interests     90
dtype: int64

Sample rows:


,ad_id,campaign_id,ad_platform,ad_type,target_gender,target_age_group,target_interests
0,1,28,Facebook,Video,Female,35-44,"art, technology"
1,2,33,Facebook,Stories,All,25-34,"travel, photography"
2,3,20,Instagram,Carousel,All,25-34,technology
3,4,28,Facebook,Stories,Female,25-34,news
4,5,24,Instagram,Image,Female,25-34,news



CAMPAIGNS
Shape: 50 rows × 6 columns

Columns:
['campaign_id', 'name', 'start_date', 'end_date', 'duration_days', 'total_budget']

Data types:
campaign_id        int64
name              object
start_date        object
end_date          object
duration_days      int64
total_budget     float64
dtype: object

Missing values:
campaign_id      0
name             0
start_date       0
end_date         0
duration_days    0
total_budget     0
dtype: int64

Full-row duplicates: 0

Unique values per column:
campaign_id      50
name             50
start_date       41
end_date         46
duration_days    33
total_budget     50
dtype: int64

Sample rows:


,campaign_id,name,start_date,end_date,duration_days,total_budget
0,1,Campaign_1_Launch,2025-05-25,2025-07-23,59,"24,021.32"
1,2,Campaign_2_Launch,2025-04-16,2025-07-07,82,"79,342.41"
2,3,Campaign_3_Winter,2025-05-04,2025-06-29,56,"14,343.25"
3,4,Campaign_4_Summer,2025-06-04,2025-08-08,65,"45,326.60"
4,5,Campaign_5_Launch,2025-07-11,2025-08-28,48,"68,376.69"



EVENTS
Shape: 67,975 rows × 7 columns

Columns:
['event_id', 'ad_id', 'user_id', 'timestamp', 'day_of_week', 'time_of_day', 'event_type']

Data types:
event_id        int64
ad_id           int64
user_id        object
timestamp      object
day_of_week    object
time_of_day    object
event_type     object
dtype: object

Missing values:
event_id       0
ad_id          0
user_id        1
timestamp      1
day_of_week    1
time_of_day    1
event_type     1
dtype: int64

Full-row duplicates: 0

Unique values per column:
event_id       67975
ad_id            200
user_id         9941
timestamp      67684
day_of_week        7
time_of_day        4
event_type         6
dtype: int64

Sample rows:


,event_id,ad_id,user_id,timestamp,day_of_week,time_of_day,event_type
0,1,197,2359b,2025-07-26 00:19:56,Saturday,Night,Like
1,2,51,f9c67,2025-06-15 08:28:07,Sunday,Morning,Share
2,3,46,5b868,2025-06-27 00:40:02,Friday,Night,Impression
3,4,166,3d440,2025-06-05 19:20:45,Thursday,Evening,Impression
4,5,52,68f1a,2025-07-22 08:30:29,Tuesday,Morning,Impression


In [4]:
# Statistical summaries for numeric columns.
for name, df in raw_dfs.items():
    print(f"\n===== {name} — numerical summary =====")
    display(df.describe(include="all").T)


===== USERS — numerical summary =====


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,10000,9950,699a5,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
user_gender,10000,3,Male,5536,NaN,NaN,NaN,NaN,NaN,NaN,NaN
user_age,"10,000.00",NaN,NaN,NaN,27.65,8.31,16.00,21.00,26.00,32.00,65.00
age_group,10000,6,25-34,4137,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,10000,10,United States,3019,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,10000,7706,West Michael,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
interests,10000,1641,fitness,283,NaN,NaN,NaN,NaN,NaN,NaN,NaN



===== ADS — numerical summary =====


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
ad_id,200.00,NaN,NaN,NaN,100.50,57.88,1.00,50.75,100.50,150.25,200.00
campaign_id,200.00,NaN,NaN,NaN,25.12,13.71,1.00,13.00,25.00,37.00,50.00
ad_platform,200,2,Facebook,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ad_type,200,4,Stories,64,NaN,NaN,NaN,NaN,NaN,NaN,NaN
target_gender,200,3,Female,83,NaN,NaN,NaN,NaN,NaN,NaN,NaN
target_age_group,200,4,35-44,54,NaN,NaN,NaN,NaN,NaN,NaN,NaN
target_interests,200,90,fashion,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN



===== CAMPAIGNS — numerical summary =====


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
campaign_id,50.00,NaN,NaN,NaN,25.50,14.58,1.00,13.25,25.50,37.75,50.00
name,50,50,Campaign_1_Launch,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
start_date,50,41,2025-05-25,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
end_date,50,46,2025-06-01,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration_days,50.00,NaN,NaN,NaN,66.04,16.38,32.00,52.50,69.50,81.75,90.00
total_budget,50.00,NaN,NaN,NaN,"50,718.48","24,576.02","7,918.04","31,105.44","48,053.65","71,600.59","98,904.66"



===== EVENTS — numerical summary =====


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
event_id,"67,975.00",NaN,NaN,NaN,"33,988.00","19,622.84",1.00,"16,994.50","33,988.00","50,981.50","67,975.00"
ad_id,"67,975.00",NaN,NaN,NaN,100.59,57.81,1.00,50.00,101.00,151.00,200.00
user_id,67974,9941,5bdf4,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
timestamp,67974,67684,2025-07-05 08:05:17,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_of_week,67974,7,Wednesday,9809,NaN,NaN,NaN,NaN,NaN,NaN,NaN
time_of_day,67974,4,Afternoon,17191,NaN,NaN,NaN,NaN,NaN,NaN,NaN
event_type,67974,6,Impression,57824,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Category distributions


In [5]:
print("EVENT TYPES")
display(events["event_type"].value_counts(dropna=False).to_frame("count"))

print("PLATFORMS")
display(ads["ad_platform"].value_counts(dropna=False).to_frame("count"))

print("AD TYPES")
display(ads["ad_type"].value_counts(dropna=False).to_frame("count"))

print("USER GENDER")
display(users["user_gender"].value_counts(dropna=False).to_frame("count"))

print("AGE GROUP")
display(users["age_group"].value_counts(dropna=False).to_frame("count"))

print("COUNTRY — top 15")
display(users["country"].value_counts(dropna=False).head(15).to_frame("count"))

EVENT TYPES


,count
event_type,
Impression,57824
Click,6725
Like,2062
Comment,687
Share,345
Purchase,331
NaN,1


PLATFORMS


,count
ad_platform,
Facebook,127
Instagram,73


AD TYPES


...

,count
user_gender,
Male,5536
Female,3443
Other,1021


AGE GROUP


,count
age_group,
25-34,4137
18-24,3116
35-44,1456
16-17,889
45-54,319
55-65,83


COUNTRY — top 15


,count
country,
United States,3019
United Kingdom,1510
Canada,1000
India,946
Germany,823
Australia,716
Brazil,607
Mexico,512
Japan,490


## 3. Primary-key validation


In [6]:
print("Duplicate event_id:", events["event_id"].duplicated().sum())
print("Duplicate ad_id:", ads["ad_id"].duplicated().sum())
print("Duplicate campaign_id:", campaigns["campaign_id"].duplicated().sum())
print("Duplicate user_id:", users["user_id"].duplicated().sum())

# The first three are required keys for the core model.
assert events["event_id"].is_unique
assert ads["ad_id"].is_unique
assert campaigns["campaign_id"].is_unique

Duplicate event_id: 0
Duplicate ad_id: 0
Duplicate campaign_id: 0
Duplicate user_id: 50


### user_id duplicates

Not renaming or appending `_dup` to anything. If two rows share a `user_id` but have different attributes, that's not a normal duplicate - it means the ID isn't reliable, so we flag it and move on.


In [7]:
duplicate_user_rows = users[
    users["user_id"].duplicated(keep=False)
].sort_values("user_id")

print("Rows involved in repeated user_id values:", len(duplicate_user_rows))
display(duplicate_user_rows.head(20))

# Compare whether the repeated IDs have conflicting attributes.
user_conflicts = (
    duplicate_user_rows
    .groupby("user_id")
    .agg(
        occurrences=("user_id", "size"),
        distinct_ages=("user_age", "nunique"),
        distinct_genders=("user_gender", "nunique"),
        distinct_countries=("country", "nunique"),
        distinct_locations=("location", "nunique"),
        distinct_interests=("interests", "nunique")
    )
    .reset_index()
)

display(user_conflicts.head(20))

print(
    "Repeated IDs with conflicting attributes:",
    (
        (user_conflicts["distinct_ages"] > 1) |
        (user_conflicts["distinct_genders"] > 1) |
        (user_conflicts["distinct_countries"] > 1) |
        (user_conflicts["distinct_locations"] > 1) |
        (user_conflicts["distinct_interests"] > 1)
    ).sum()
)

Rows involved in repeated user_id values: 100


,user_id,user_gender,user_age,age_group,country,location,interests
5384,02ad5,Female,21,18-24,Brazil,New Adam,"technology, gaming, sports"
7258,02ad5,Female,38,35-44,Canada,North Stacy,"art, lifestyle, food"
9850,0b8c2,Male,27,25-34,United States,Harrisland,"technology, sports"
6045,0b8c2,Male,27,25-34,Canada,West Caroline,"art, technology"
2570,0ebd6,Male,32,25-34,Canada,Port Melissaport,sports
1029,0ebd6,Male,28,25-34,United States,South Jason,health
7091,11bbd,Male,29,25-34,Australia,Gonzalezstad,"lifestyle, health, fashion"
2398,11bbd,Other,25,25-34,United Kingdom,West Donaldchester,"finance, technology, lifestyle"
4896,15f7b,Male,18,18-24,United States,East Dawn,"art, sports"
3987,15f7b,Male,35,35-44,United States,North Susanville,"photography, gaming, lifestyle"


,user_id,occurrences,distinct_ages,distinct_genders,distinct_countries,distinct_locations,distinct_interests
0,02ad5,2,2,1,2,2,2
1,0b8c2,2,1,1,2,2,2
2,0ebd6,2,2,1,2,2,2
3,11bbd,2,2,2,2,2,2
4,15f7b,2,2,1,1,2,2
5,168cf,2,2,2,2,2,2
6,1dc9b,2,2,2,2,2,2
7,23c32,2,2,1,2,2,2
8,2af0c,2,2,2,2,2,2
9,2cff5,2,2,2,2,2,2


Repeated IDs with conflicting attributes: 50


All 50 duplicate `user_id` groups turn out to have conflicting attributes, so this isn't a couple of harmless repeats - the ID genuinely can't be trusted. Decision: keep the original values, flag the collisions, and don't join `events` to `users` in the core model.


In [8]:
users["user_id_collision"] = users["user_id"].duplicated(keep=False)

print("Users with collision-prone IDs:",
      users["user_id_collision"].sum())

Users with collision-prone IDs: 100


## 4. Data types


In [9]:
# Work on copies so the original raw objects remain conceptually untouched.
users_clean = users.copy()
ads_clean = ads.copy()
campaigns_clean = campaigns.copy()
events_clean = events.copy()

events_clean["timestamp"] = pd.to_datetime(
    events_clean["timestamp"],
    errors="coerce"
)

campaigns_clean["start_date"] = pd.to_datetime(
    campaigns_clean["start_date"],
    errors="coerce"
)

campaigns_clean["end_date"] = pd.to_datetime(
    campaigns_clean["end_date"],
    errors="coerce"
)

print("Invalid timestamps:", events_clean["timestamp"].isna().sum())
print("Invalid start dates:", campaigns_clean["start_date"].isna().sum())
print("Invalid end dates:", campaigns_clean["end_date"].isna().sum())

Invalid timestamps: 1
Invalid start dates: 0
Invalid end dates: 0


## 5. Text cleanup


In [10]:
# Strip unnecessary surrounding whitespace.
for col in ["user_gender", "country", "interests"]:
    users_clean[col] = users_clean[col].astype("string").str.strip()

for col in ["ad_platform", "ad_type", "target_gender", "target_age_group", "target_interests"]:
    ads_clean[col] = ads_clean[col].astype("string").str.strip()

for col in ["event_type", "day_of_week", "time_of_day"]:
    events_clean[col] = events_clean[col].astype("string").str.strip()

# Standardize categorical casing where appropriate.
users_clean["user_gender"] = users_clean["user_gender"].str.title()
ads_clean["target_gender"] = ads_clean["target_gender"].str.title()
ads_clean["ad_platform"] = ads_clean["ad_platform"].str.title()
ads_clean["ad_type"] = ads_clean["ad_type"].str.title()
events_clean["event_type"] = events_clean["event_type"].str.title()
events_clean["time_of_day"] = events_clean["time_of_day"].str.title()

# Normalize comma-separated interest text.
users_clean["interests"] = (
    users_clean["interests"]
    .str.lower()
    .str.replace(r"\s*,\s*", ", ", regex=True)
)

ads_clean["target_interests"] = (
    ads_clean["target_interests"]
    .str.lower()
    .str.replace(r"\s*,\s*", ", ", regex=True)
)

## 6. Category & business-rule checks


In [11]:
expected_event_types = {
    "Impression", "Click", "Like", "Comment", "Share", "Purchase"
}
expected_platforms = {"Facebook", "Instagram"}
expected_ad_types = {"Image", "Video", "Carousel", "Stories"}

invalid_events = set(events_clean["event_type"].dropna().unique()) - expected_event_types
invalid_platforms = set(ads_clean["ad_platform"].dropna().unique()) - expected_platforms
invalid_ad_types = set(ads_clean["ad_type"].dropna().unique()) - expected_ad_types

print("Unexpected event types:", invalid_events)
print("Unexpected platforms:", invalid_platforms)
print("Unexpected ad types:", invalid_ad_types)

assert not invalid_events
assert not invalid_platforms
assert not invalid_ad_types

Unexpected event types: set()
Unexpected platforms: set()
Unexpected ad types: set()


In [12]:
# Numeric / date sanity checks.
print("Non-positive event IDs:", (events_clean["event_id"] <= 0).sum())
print("Non-positive ad IDs:", (ads_clean["ad_id"] <= 0).sum())
print("Non-positive campaign IDs:", (campaigns_clean["campaign_id"] <= 0).sum())
print("Non-positive campaign budgets:", (campaigns_clean["total_budget"] <= 0).sum())

invalid_campaign_dates = campaigns_clean[
    campaigns_clean["end_date"] < campaigns_clean["start_date"]
]

print("Campaigns with end_date before start_date:",
      len(invalid_campaign_dates))

print("Users with negative age:", (users_clean["user_age"] < 0).sum())


Non-positive event IDs: 0
Non-positive ad IDs: 0
Non-positive campaign IDs: 0
Non-positive campaign budgets: 0
Campaigns with end_date before start_date: 0
Users with negative age: 0


## 7. Foreign-key validation


In [13]:
invalid_event_ads = ~events_clean["ad_id"].isin(ads_clean["ad_id"])
invalid_ad_campaigns = ~ads_clean["campaign_id"].isin(campaigns_clean["campaign_id"])

print("Events with invalid ad_id:", invalid_event_ads.sum())
print("Ads with invalid campaign_id:", invalid_ad_campaigns.sum())

assert invalid_event_ads.sum() == 0
assert invalid_ad_campaigns.sum() == 0

Events with invalid ad_id: 0
Ads with invalid campaign_id: 0


## 8. Check the derived time fields


In [14]:
# Recalculate day of week from timestamp.
expected_day = events_clean["timestamp"].dt.day_name()

day_mismatches = (
    expected_day != events_clean["day_of_week"]
).fillna(False)

print("day_of_week mismatches:", day_mismatches.sum())

day_of_week mismatches: 0


In [15]:
def derive_time_of_day(hour):
    if pd.isna(hour):
        return np.nan
    if 6 <= hour < 12:
        return "Morning"
    if 12 <= hour < 18:
        return "Afternoon"
    if 18 <= hour < 24:
        return "Evening"
    return "Night"

expected_time = events_clean["timestamp"].dt.hour.map(derive_time_of_day)

time_mismatches = (
    expected_time != events_clean["time_of_day"]
).fillna(False)

print("time_of_day mismatches:", time_mismatches.sum())

time_of_day mismatches: 0


Both come back at 0 mismatches, so the supplied `day_of_week` and `time_of_day` columns are trustworthy - no need to overwrite them. If a future run of this notebook shows nonzero mismatches, don't just overwrite the columns - check why before deciding what to do.


## 9. Campaign vs. event date consistency


In [16]:
date_check = (
    events_clean
    .merge(
        ads_clean[["ad_id", "campaign_id"]],
        on="ad_id",
        how="left",
        validate="many_to_one"
    )
    .merge(
        campaigns_clean[["campaign_id", "start_date", "end_date"]],
        on="campaign_id",
        how="left",
        validate="many_to_one"
    )
)

date_check["within_campaign_window"] = (
    (date_check["timestamp"] >= date_check["start_date"]) &
    (date_check["timestamp"] <= date_check["end_date"])
)

outside_window = ~date_check["within_campaign_window"]

print("Events outside campaign window:", outside_window.sum())
print(
    "Percentage outside:",
    round(outside_window.mean() * 100, 2),
    "%"
)

Events outside campaign window: 38294
Percentage outside: 56.34 %


### What to do about the date mismatch

56.35% of events happen outside their campaign's start/end window. That's not a rounding error, that's more than half the dataset - so campaign dates clearly aren't a reliable way to filter or bucket events by time.

We're keeping both: event timestamps stay as the real time dimension for EDA and the funnel, campaign dates stay as metadata only. Nothing gets deleted or backfilled here - this is a known limitation of the source data, not something to quietly patch.


## 10. Cardinality checks


In [17]:
# One campaign can have many ads.
ads_per_campaign = (
    ads_clean.groupby("campaign_id")["ad_id"]
    .nunique()
)

print("Campaigns with no ads:",
      campaigns_clean.loc[
          ~campaigns_clean["campaign_id"].isin(ads_clean["campaign_id"]),
          "campaign_id"
      ].tolist())

print("Min ads per represented campaign:", ads_per_campaign.min())
print("Max ads per represented campaign:", ads_per_campaign.max())

# Validate the actual core joins without bringing users into the model.
events_ads = events_clean.merge(
    ads_clean[["ad_id", "campaign_id", "ad_platform", "ad_type"]],
    on="ad_id",
    how="left",
    validate="many_to_one"
)

events_ads_campaigns = events_ads.merge(
    campaigns_clean[["campaign_id", "name"]],
    on="campaign_id",
    how="left",
    validate="many_to_one"
)

print("Core analytical rows:", len(events_ads_campaigns))
print("Original event rows:", len(events_clean))

assert len(events_ads_campaigns) == len(events_clean)

Campaigns with no ads: [16, 43]
Min ads per represented campaign: 1
Max ads per represented campaign: 8
Core analytical rows: 67975
Original event rows: 67975


## 11. Post-cleaning audit


In [18]:
clean_dfs = {
    "USERS": users_clean,
    "ADS": ads_clean,
    "CAMPAIGNS": campaigns_clean,
    "EVENTS": events_clean
}

for name, df in clean_dfs.items():
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Missing values:", df.isna().sum().sum())
    print("Full-row duplicates:", df.duplicated().sum())


===== USERS =====
Shape: (10000, 8)
Missing values: 0
Full-row duplicates: 0

===== ADS =====
Shape: (200, 7)
Missing values: 0
Full-row duplicates: 0

===== CAMPAIGNS =====
Shape: (50, 6)
Missing values: 0
Full-row duplicates: 0

===== EVENTS =====
Shape: (67975, 7)
Missing values: 5
Full-row duplicates: 0


In [19]:
validation_summary = pd.DataFrame({
    "Check": [
        "Duplicate event_id",
        "Duplicate ad_id",
        "Duplicate campaign_id",
        "Invalid event → ad links",
        "Invalid ad → campaign links",
        "Invalid event timestamps",
        "Invalid campaign start dates",
        "Invalid campaign end dates",
        "Campaign end before start",
        "day_of_week mismatches",
        "time_of_day mismatches"
    ],
    "Count": [
        events_clean["event_id"].duplicated().sum(),
        ads_clean["ad_id"].duplicated().sum(),
        campaigns_clean["campaign_id"].duplicated().sum(),
        invalid_event_ads.sum(),
        invalid_ad_campaigns.sum(),
        events_clean["timestamp"].isna().sum(),
        campaigns_clean["start_date"].isna().sum(),
        campaigns_clean["end_date"].isna().sum(),
        len(invalid_campaign_dates),
        day_mismatches.sum(),
        time_mismatches.sum()
    ]
})

display(validation_summary)

,Check,Count
0,Duplicate event_id,0
1,Duplicate ad_id,0
2,Duplicate campaign_id,0
3,Invalid event → ad links,0
4,Invalid ad → campaign links,0
5,Invalid event timestamps,1
6,Invalid campaign start dates,0
7,Invalid campaign end dates,0
8,Campaign end before start,0
9,day_of_week mismatches,0


## 12. Export clean tables


In [20]:
# These are the files used by the next notebook and Power BI.
users_clean.to_csv("users_clean.csv", index=False)
ads_clean.to_csv("ads_clean.csv", index=False)
campaigns_clean.to_csv("campaigns_clean.csv", index=False)
events_clean.to_csv("events_clean.csv", index=False)

print("Clean files exported:")
print("- users_clean.csv")
print("- ads_clean.csv")
print("- campaigns_clean.csv")
print("- events_clean.csv")

Clean files exported:
- users_clean.csv
- ads_clean.csv
- campaigns_clean.csv
- events_clean.csv


## Data model

One campaign has many ads, one ad has many events. `campaigns_clean -> ads_clean -> events_clean`.

`users_clean` is exported for reference but sits outside this model - `user_id` isn't a reliable key.

